# W(5,2) = 178, as a machine-checked Lean theorem

Take the numbers 1 through 178. Color each one red or blue, any way you like.

Claim: no matter how you do it, five of them, evenly spaced, end up the same color.

"Evenly spaced" means an arithmetic progression — numbers like 3, 7, 11, 15,
19 (start 3, step 4, five terms). "Same color" means monochromatic.

This is the van der Waerden number **W(5,2) = 178**: the smallest N such that
every 2-coloring of {1,...,N} is forced to contain a monochromatic 5-term
arithmetic progression. Below 178 you can dodge it — there's a coloring of
{1,...,177} with no monochromatic 5-AP at all. At 178 you're stuck: every
coloring has one.

"No matter how you do it" is the hard part. It's a claim about all 2^178
colorings, not one clever example. This notebook walks through how that claim
became a Lean theorem — the tool mathematicians use to check proofs by
machine, not by reading them.

First, a quick look at why the claim is non-obvious: a random coloring
usually fails almost immediately.

In [1]:
import random

def find_mono_ap(coloring, n, length):
    # first monochromatic AP of `length` terms in a coloring of {1,...,n}
    for d in range(1, n):
        for a in range(1, n - (length - 1) * d + 1):
            positions = [a + k * d for k in range(length)]
            colors = {coloring[p] for p in positions}
            if len(colors) == 1:
                return positions
    return None

random.seed(0)
N = 178
coloring = {i: random.choice(["red", "blue"]) for i in range(1, N + 1)}

ap = find_mono_ap(coloring, N, 5)
print(f"random coloring of 1..{N}:")
print(f"  positions {ap} are all {coloring[ap[0]]} -- a monochromatic AP")
print("  (step", ap[1] - ap[0], ")")

random coloring of 1..178:
  positions [4, 5, 6, 7, 8] are all blue -- a monochromatic AP
  (step 1 )


## The easy half: a coloring of 1..177 that works

W(5,2) = 178 bundles two separate claims:

1. {1,...,177} **can** be colored with no monochromatic 5-AP (the lower bound).
2. {1,...,178} **cannot** (the upper bound).

Claim 1 is easy to check once you have the coloring: just look at it. The
Lean theorem embeds the exact coloring it uses, as a literal list of 177
True/False values, in `lean/W177Witness.lean`. Below we read that file
directly -- not a copy, the actual array the proof is built from -- and
check it ourselves, in plain Python, no SAT solver involved.

In [2]:
import re

with open("lean/W177Witness.lean") as f:
    lean_src = f.read()

# Pull the literal list straight out of: def w177True : List Nat := [3, 4, 5, ...]
match = re.search(r"def w177True : List Nat :=\s*\[(.*?)\]", lean_src, re.S)
blue_positions = {int(x) for x in re.findall(r"\d+", match.group(1))}
print(f"parsed {len(blue_positions)} 'blue' positions out of 177 from the Lean source")

N, LEN = 177, 5
color = lambda i: "blue" if i in blue_positions else "red"

checked = 0
for d in range(1, N):
    for a in range(1, N - (LEN - 1) * d + 1):
        positions = [a + k * d for k in range(LEN)]
        colors = {color(p) for p in positions}
        assert len(colors) == 2, f"monochromatic AP found: {positions}"
        checked += 1

print(f"checked all {checked} APs of length {LEN} in 1..{N}: none monochromatic")

parsed 88 'blue' positions out of 177 from the Lean source
checked all 3828 APs of length 5 in 1..177: none monochromatic


## The hard half, layer 1: turning "no coloring works" into a SAT problem

Claim 2 -- nothing colors {1,...,178} safely -- can't be checked by looking
at one example. It's a claim about *every* one of the 2^178 colorings. That's
where SAT (boolean satisfiability) comes in.

The encoding (`code/vdw_sat.py`, function `encode`) uses one boolean variable
per position: `x_i = True` means position `i` is blue, `False` means red.
For every 5-term AP in {1,...,178}, two clauses go in:

- `(x_a OR x_{a+d} OR ... OR x_{a+4d})` -- forbids all five being red (all False)
- `(NOT x_a OR NOT x_{a+d} OR ... OR NOT x_{a+4d})` -- forbids all five being blue

A clause means "at least one of these must be true." If a solver finds an
assignment that satisfies every clause, that assignment *is* a valid
coloring -- no monochromatic AP anywhere. If **no** assignment satisfies
every clause (UNSAT), no valid coloring exists at all. UNSAT is exactly the
upper-bound claim.

Below is the same encoding at toy scale -- W(3,2)=9, so 3-term APs, N=9 --
showing actual clauses next to the APs they came from.

In [3]:
import sys
sys.path.insert(0, "code")
import vdw_sat

clauses, nvars = vdw_sat.encode([3, 3], 9)  # toy: 3-term APs, N=9
aps = list(vdw_sat.ap_starts(9, 3))
print(f"toy instance: {nvars} variables, {len(clauses)} clauses ({len(aps)} APs, x2)\n")

for a, d in [aps[0], aps[-1]]:
    ap = [a + k * d for k in range(3)]
    pos_clause = next(c for c in clauses if set(c) == set(ap))
    neg_clause = next(c for c in clauses if set(c) == {-p for p in ap})
    print(f"AP {ap} (a={a}, d={d}):")
    print(f"  {pos_clause}  -- forbids all-red")
    print(f"  {neg_clause}  -- forbids all-blue\n")

toy instance: 9 variables, 32 clauses (16 APs, x2)

AP [1, 2, 3] (a=1, d=1):
  [1, 2, 3]  -- forbids all-red
  [-1, -2, -3]  -- forbids all-blue

AP [1, 5, 9] (a=1, d=4):
  [1, 5, 9]  -- forbids all-red
  [-1, -5, -9]  -- forbids all-blue



## Layer 2: cube-and-conquer -- splitting one huge problem into thousands of small ones

A single SAT solver call on the real instance (178 variables, 7744 clauses)
is too slow to finish directly -- the search tree is too deep. The build
that produced this theorem split it instead.

`march_cu` (a look-ahead solver, in `tools/CnC/`) doesn't try to solve the
formula -- it picks a handful of variables and case-splits on them, producing
a list of "cubes": partial assignments. Adding a cube's literals as extra
unit clauses turns the big formula into a smaller, easier sub-problem. Solve
each cube's sub-problem independently -- in parallel, e.g. across separate
GitHub Actions jobs -- and if **every** cube's sub-problem is UNSAT, the
whole formula is UNSAT. That's valid because the cubes, by construction,
cover every possible assignment: every combination of True/False for the
split variables shows up as some cube's partial assignment. Miss a branch
and the proof is wrong, so the split's completeness has to be checked too
(that's what the "cover" step in the recipe below verifies).

For the real N=178 instance this produces 3627 cubes (`lean/BUILD.md`).
Below is the same tool, run fresh on a tiny toy (N=15) so you can see what a
cube actually looks like.

In [4]:
import subprocess, tempfile, pathlib

outdir = tempfile.mkdtemp()
result = subprocess.run(
    ["python3", "code/vdw_cnc.py", "split",
     "--lengths", "5", "5", "--N", "15",
     "--outdir", outdir, "--march-opts", "-d 5"],
    capture_output=True, text=True,
)
print(result.stdout.strip())

cubes_file = next(pathlib.Path(outdir).glob("*.cubes"))
cube_lines = cubes_file.read_text().splitlines()
print(f"\n{len(cube_lines)} cubes produced. First one:")
print(f"  {cube_lines[0]}")
print("  an 'a ... 0' line: a partial assignment (one literal per split")
print("  variable). Solving the ORIGINAL formula plus these units as extra")
print("  unit clauses is the sub-problem for this one cube.")

split: /Users/abigailhaddad/Documents/repos/proof/tools/CnC/march_cu/march_cu /var/folders/px/34g351m146v2cvs_ct7jf3q00000gn/T/tmpzm9tobq_/cnc_w5_5_N15.cnf -o /var/folders/px/34g351m146v2cvs_ct7jf3q00000gn/T/tmpzm9tobq_/cnc_w5_5_N15.cubes -d 5
  split: 15 vars, 42 clauses -> 32 cubes in 0.0s

32 cubes produced. First one:
  a -9 -6 -11 -7 -5 0
  an 'a ... 0' line: a partial assignment (one literal per split
  variable). Solving the ORIGINAL formula plus these units as extra
  unit clauses is the sub-problem for this one cube.


## Layer 3: certificates -- proving each UNSAT sub-problem, and checking the proof

A SAT solver run in UNSAT mode doesn't have to just say "no" -- it can emit
an LRAT proof: a trace of every clause it derived on the way to a
contradiction, with each step justified by which earlier clauses imply it.
The format is simple enough that a small, independently-written checker can
verify it line by line -- much simpler than the search that produced it, so
you don't have to trust the solver, only the (tiny, auditable) checker.

Below is the actual certificate for the W(3,2)=9 toy, solved directly with
no cube-splitting needed (`lean/vdw_3_9.lrat`, paired with `lean/vdw_3_9.cnf`
from the previous section).

In [5]:
print(open("lean/vdw_3_9.lrat").read())

33 -6 -8 -9 0 23 27 31 13 3 32 0
34 -8 -9 0 23 33 5 30 32 1 9 19 0
34 d 23 0
35 7 -9 0 34 6 31 10 20 32 13 0
36 -6 -9 0 34 35 28 14 31 25 3 0
36 d 33 0
37 -9 0 34 35 28 14 36 11 29 18 8 0
38 5 6 0 37 16 12 4 29 0
39 -5 -3 0 26 24 19 13 0
39 d 26 0
40 6 0 37 15 38 39 0
41 -5 0 40 37 20 39 21 2 7 30 0
42 1 0 41 37 16 0
43 7 0 41 37 12 0
44 -4 0 42 43 29 0
45 -8 0 43 40 22 0
46 3 0 44 41 3 0
47 2 0 45 41 14 0
48 0 47 46 42 17 0



Reading it: each numbered line (`33 -6 -8 -9 0 23 27 31 13 3 32 0`) adds one
new derived clause and lists the earlier clause numbers that justify it. A
`d` line (`34 d 23 0`) deletes a clause the checker no longer needs, keeping
its working memory small. The checker replays every line in order and
confirms the last derived clause is empty -- a direct contradiction, which is
only reachable if the original formula was genuinely UNSAT.

Running the actual checker (`tools/drat-trim/lrat-check`, part of this
repo's toolchain) against both files:

In [6]:
import subprocess, os

checker = "tools/drat-trim/lrat-check"
if os.access(checker, os.X_OK):
    result = subprocess.run([checker, "lean/vdw_3_9.cnf", "lean/vdw_3_9.lrat"],
                             capture_output=True, text=True)
    print(result.stdout)
else:
    print(f"{checker} not executable in this environment -- skipping the run.")

c parsed a formula with 9 variables and 32 clauses
c VERIFIED
c allocated 1024 216 90
c Added clauses = 48.  Deleted clauses = 3.  Max live clauses = 45
c verification time = 0.00 secs



## Layer 4: Lean -- the actual theorem statement

LRAT certificates prove individual CNF files UNSAT. They don't, by
themselves, say anything about *colorings* -- that connection (does this CNF
really encode "no valid 2-coloring exists"?) still has to be argued, and for
the claim to count as a genuine theorem, that argument has to be checked by
Lean's kernel, not just read by a person and trusted.

Here is the actual statement, from `lean/VdW.lean`, verbatim:

```lean
def vdwNumber2 (len W : Nat) : Prop :=
  hasAPFree2 len (W - 1) ∧ ¬hasAPFree2 len W

def hasAPFree2 (len n : Nat) : Prop := ∃ f : Nat → Bool, isAPFree2 len n f

def isAPFree2 (len n : Nat) (f : Nat → Bool) : Prop :=
  ∀ ap ∈ apTuples n len, (∃ p ∈ ap, f p = true) ∧ (∃ p ∈ ap, f p = false)

def apTuples (n len : Nat) : List (List Nat) :=
  -- all a, a+d, …, a+(len-1)·d with d ≥ 1, a ≥ 1, a+(len-1)·d ≤ n
```

One sentence per line:

- `apTuples n len` -- the list of every length-`len` arithmetic progression
  that fits inside {1,...,n}, exactly as defined (all valid `a`, `d` pairs).
- `isAPFree2 len n f` -- coloring `f` is valid for length `len` on {1,...,n}:
  every AP in the list contains at least one `True` position and at least
  one `False` position (so it's never monochromatic).
- `hasAPFree2 len n` -- some valid coloring `f` exists for {1,...,n}.
- `vdwNumber2 len W` -- the full claim: a valid coloring exists for
  {1,...,W-1} (the lower bound, `w177_good`), and no valid coloring exists
  for {1,...,W} (the upper bound, built from the SAT certificates above).

The actual theorem: `vdw_5_2 : vdwNumber2 5 178`. No CNF, no clause, no
solver appears anywhere in this statement -- it's a claim purely about
functions from `Nat` to `Bool`. Everything about SAT encoding, cubes, and
LRAT certificates lives in the *proof*, not the statement, and Lean's kernel
checked that the proof actually establishes it.

`#print axioms vdw_5_2` lists every non-constructive step the proof leans
on, verbatim:

```
propext
Classical.choice
Quot.sound
w177_good._native.native_decide.ax_1_1
base_covers._native.native_decide.ax_1_1
chunk1_ok._native.native_decide.ax_1_1
chunk2_ok._native.native_decide.ax_1_1
… (chunk3_ok … chunk73_ok, 73 total)
coverThm._native.native_decide.ax_1_1
```

`propext`, `Classical.choice`, `Quot.sound` are standard -- nearly every
nontrivial Lean proof uses them. The 76 `native_decide` instances are each a
compiled boolean check (the witness coloring, the SAT-to-Lean bridge, and
each of the 73 proof chunks) -- Lean compiles the check to native code, runs
it, and trusts the result, the same trust model Lean's `bv_decide` tactic
uses. No `sorry` anywhere.

**The trust chain, plainly: you are trusting Lean's kernel and Lean's
compiler. You are NOT trusting our Python, the SAT solvers, or the AI that
wrote everything.** The Python (`vdw_sat.py`, `vdw_cnc.py`) generated the
CNF and ran the solvers, but the Python's output was never taken on faith --
every claim it produced (the witness coloring, the UNSAT certificates) was
re-derived and re-checked inside Lean from first principles. If the Python
had a bug that produced a bogus witness or a bogus certificate, Lean's
kernel would have rejected it, not silently accepted it.

## Reproduce it yourself (~20 minutes)

Condensed from `lean/BUILD.md`, which has the full recipe and gotchas.

1. Clone `lrat-catcher` and pin the exact commit this was built against:
   ```
   git clone https://github.com/leansolving/lrat-catcher.git
   cd lrat-catcher
   git checkout 4ec2168b810636e789da3349ab3e670af338187c
   ```
   (`lean-toolchain` pins `leanprover/lean4:v4.30.0`; `elan` installs it
   automatically on first `lake build`.)

2. Drop this repo's `lean/*.lean`, `lean/vdw_3_9.{cnf,lrat}`, and
   `lean/gen_vdw.patch` into the paths listed in `BUILD.md`'s file-placement
   table, and `git apply gen_vdw.patch` at the clone root.

3. Regenerate the dress-rehearsal artifact `Generated/W52b` (~20-25 min):
   split the N=178 instance into cubes with `march_cu`, export per-cube leaf
   CNFs, solve each leaf with `cadical --lrat --no-binary --no-factor`
   (parallel, ~23s wall), compose the per-cube LRAT certificates plus a
   cover-completeness check into 73 Lean chunk modules, then `./build.sh`.

4. `lake build LRATCatcher.Tests.VdW52Test` -- builds the bridge and the
   final theorem (~21s). `#print axioms vdw_5_2` to see the axiom list
   yourself.

This proof repo's own commit for this theorem: `b1c8c39`.

## Honest context

W(5,2) = 178 is not a new number -- it's been known, computationally, since
1978. What's new here is not the value, it's the *verification*: turning
"a solver said UNSAT" into a statement Lean's kernel checked from the ground
up, with an explicit, minimal, and readable list of what you have to trust
to believe it.

The bigger landscape: exact diagonal van der Waerden numbers are known only
up to W(6,2) = 1132 (Kouril's 2008 thesis, no machine-checkable certificate
found anywhere), and W(7,2) is considered out of reach with current methods
("never," in Kouril's own assessment). A mathematician we consulted made the
useful framing point: the genuinely hard direction in this whole area isn't
finding a clever coloring that avoids a pattern (a lower bound) -- it's
proving that literally *every* coloring fails (an upper bound). That's the
direction this theorem is in.